# Engine-Component Detection: Dataset Construction and YOLOv11 Variant Study

This notebook reproduces the full pipeline behind the consolidated eight-class
engine-component detection dataset and the in-domain YOLOv11 variant study.

It is organised in seven parts:
1. **Dataset creation** — download a public Roboflow source, filter it to the eight
   target classes, and merge in a Kaggle engine-bay source under a common taxonomy.
2. **Leakage check and clean split** — detect near-duplicate leakage with perceptual
   hashing and re-split the data by source identity.
3. **Training** — fine-tune YOLOv11m from COCO-pretrained weights.
4. **Evaluation** — overall metrics, per-class behaviour and error analysis.
5. **Variant comparison** — train the n, s and m variants under identical conditions.
6. **Augmentation ablation** — remove one augmentation family at a time.
7. **Leakage on/off experiment** — quantify how much uncorrected leakage inflates metrics.

The eight target classes are: alternator, battery, radiator, ignition coil, starter,
fuel injector, oil filter and spark plug.

## Part 0 — Environment Setup

We start by checking the assigned GPU, installing the two external dependencies
(`ultralytics` for YOLOv11 and `imagehash` for perceptual hashing), and mounting
Google Drive so that intermediate artefacts can be saved and reused across sessions.
The first cell keeps the Colab session alive during long training runs.

In [ ]:
# Keep the Colab session alive during long-running training cells.
import IPython
from google.colab import output

display(IPython.display.Javascript('''
function ClickConnect(){
    console.log("Keeping alive...");
    document.querySelector("colab-connect-button").click()
}
setInterval(ClickConnect, 60000)
'''))
print("Anti-disconnect enabled")

In [ ]:
# Check the GPU assigned to this Colab session.
!nvidia-smi

In [ ]:
# Install the two external dependencies: Ultralytics (YOLOv11) and imagehash (pHash).
!pip install ultralytics imagehash roboflow pyyaml -q

In [ ]:
# Mount Google Drive to persist datasets, weights and exported figures.
from google.colab import drive
drive.mount('/content/drive')

## Part 1 — Dataset Creation

No public detection dataset matches our eight-class diagnostic-and-repair taxonomy, so the
dataset is consolidated from two complementary public sources: a Roboflow automotive-parts
dataset (`team-data/car-parts-ybiev`) and a Kaggle engine-bay dataset.

This part uses a **use-or-build** strategy. The next cell checks Drive for a prebuilt
`dataset.zip`:

- if it exists, the dataset is unpacked from it directly (fast path, no API key needed);
- if it does not exist, sections 1.1–1.3 rebuild the dataset from the two sources and the
  final section repacks it into `dataset.zip` on Drive, so subsequent runs take the fast path.

Run the check cell first. If it reports that the dataset was loaded from the zip, skip
ahead to Part 2. Otherwise, run sections 1.1–1.3 and the repack cell.

### 1.0 — Check for a prebuilt dataset

In [ ]:
# Use-or-build: try the prebuilt dataset.zip first; otherwise we build it below.
import os, shutil

DATASET_DIR = "/content/data/dataset"
ZIP_PATH = "/content/drive/MyDrive/dataset.zip"   # where the prebuilt dataset lives
DATASET_READY = False

if os.path.exists(ZIP_PATH):
    if os.path.exists(DATASET_DIR):
        shutil.rmtree(DATASET_DIR)
    os.makedirs(DATASET_DIR, exist_ok=True)
    shutil.copy(ZIP_PATH, "/content/dataset.zip")
    !unzip -q -o /content/dataset.zip -d {DATASET_DIR}
    DATASET_READY = True
    print("Loaded dataset from prebuilt dataset.zip")
    print("=> You can skip sections 1.1-1.3 and go to Part 2.")
    !ls {DATASET_DIR}
else:
    print("No dataset.zip found on Drive.")
    print("=> Run sections 1.1-1.3 to build it, then the repack cell (1.4).")

### 1.1 — Download the Roboflow baseline

The Roboflow source is downloaded in YOLOv11 format through the Roboflow API. The API key
is read from the environment; set `ROBOFLOW_API_KEY` before running this cell. The download
produces a standard YOLO directory with `train/valid/test` splits and a `data.yaml`
describing the original (broad) class list. Run this only if no prebuilt dataset was loaded
in 1.0.

In [ ]:
# Run only if DATASET_READY is False (no prebuilt zip was found).
if not DATASET_READY:
    os.environ["ROBOFLOW_API_KEY"] = os.environ.get("ROBOFLOW_API_KEY", "YOUR_KEY_HERE")
    from roboflow import Roboflow

    BASELINE_DIR = "/content/data/baseline"
    os.makedirs(BASELINE_DIR, exist_ok=True)

    rf = Roboflow(api_key=os.environ["ROBOFLOW_API_KEY"])
    project = rf.workspace("team-data").project("car-parts-ybiev")
    version = project.versions()[0]  # latest version
    version.download(
        model_format="yolov11",
        location=os.path.join(BASELINE_DIR, "car-parts"),
        overwrite=True,
    )

    dataset_path = os.path.join(BASELINE_DIR, "car-parts")
    for split in ["train", "valid", "test"]:
        images_dir = os.path.join(dataset_path, split, "images")
        if os.path.exists(images_dir):
            n = len([f for f in os.listdir(images_dir) if not f.startswith(".")])
            print(f"  {split:6s}: {n} images")
else:
    print("Dataset already loaded from zip; skipping Roboflow download.")

### 1.2 — Filter to the eight target classes and remap IDs

The Roboflow source contains many more categories than we need. This step keeps only the
eight target classes and remaps their original class IDs onto our fixed taxonomy
(`alternator=0`, `battery=1`, `radiator=2`, `ignition_coil=3`, `starter=4`,
`fuel_injector=5`, `oil_filter=6`, `spark_plug=7`). Each label file is rewritten with the
new IDs, and only images that retain at least one target annotation are copied. The Roboflow
`valid` split is renamed to `val` to match the rest of the pipeline.

In [ ]:
if not DATASET_READY:
    import yaml

    TARGET_CLASSES = {
        "ALTERNATOR": 0, "BATTERY": 1, "RADIATOR": 2, "IGNITION COIL": 3,
        "STARTER": 4, "FUEL INJECTOR": 5, "OIL FILTER": 6, "SPARK PLUG": 7,
    }
    OUR_CLASS_NAMES = [
        "alternator", "battery", "radiator", "ignition_coil",
        "starter", "fuel_injector", "oil_filter", "spark_plug",
    ]
    BASELINE_CARPARTS = os.path.join("/content/data/baseline", "car-parts")

    def load_baseline_classes(baseline_dir):
        with open(os.path.join(baseline_dir, "data.yaml")) as f:
            data = yaml.safe_load(f)
        names = data.get("names", [])
        if isinstance(names, list):
            return {i: name for i, name in enumerate(names)}
        return {int(k): v for k, v in names.items()}

    def filter_label_file(label_path, id_to_name):
        out = []
        with open(label_path) as f:
            for line in f:
                parts = line.strip().split()
                if len(parts) < 5:
                    continue
                name = id_to_name.get(int(parts[0]), "")
                if name in TARGET_CLASSES:
                    out.append(f"{TARGET_CLASSES[name]} {' '.join(parts[1:])}")
        return out

    def process_split(split, baseline_dir, output_dir, id_to_name):
        out_split = "val" if split == "valid" else split
        src_images = os.path.join(baseline_dir, split, "images")
        src_labels = os.path.join(baseline_dir, split, "labels")
        dst_images = os.path.join(output_dir, "images", out_split)
        dst_labels = os.path.join(output_dir, "labels", out_split)
        os.makedirs(dst_images, exist_ok=True)
        os.makedirs(dst_labels, exist_ok=True)
        if not os.path.exists(src_labels):
            return 0
        kept = 0
        for label_file in os.listdir(src_labels):
            if not label_file.endswith(".txt"):
                continue
            filtered = filter_label_file(os.path.join(src_labels, label_file), id_to_name)
            if not filtered:
                continue
            stem = os.path.splitext(label_file)[0]
            for ext in [".jpg", ".jpeg", ".png", ".webp"]:
                img = os.path.join(src_images, stem + ext)
                if os.path.exists(img):
                    shutil.copy2(img, os.path.join(dst_images, stem + ext))
                    with open(os.path.join(dst_labels, label_file), "w") as f:
                        f.write("\n".join(filtered) + "\n")
                    kept += 1
                    break
        return kept

    if os.path.exists(DATASET_DIR):
        shutil.rmtree(DATASET_DIR)

    id_to_name = load_baseline_classes(BASELINE_CARPARTS)
    print("Baseline class mapping:")
    for name, our_id in TARGET_CLASSES.items():
        present = [bid for bid, bname in id_to_name.items() if bname == name]
        print(f"  {name:16s} -> {our_id} (baseline id: {present[0] if present else 'MISSING'})")

    total = 0
    for split in ["train", "valid", "test"]:
        kept = process_split(split, BASELINE_CARPARTS, DATASET_DIR, id_to_name)
        total += kept
        print(f"  {split:6s}: kept {kept} images")
    print(f"Total after filtering: {total} images")

    with open(os.path.join(DATASET_DIR, "data.yaml"), "w") as f:
        yaml.dump({
            "path": DATASET_DIR, "train": "images/train", "val": "images/val",
            "test": "images/test", "nc": len(OUR_CLASS_NAMES), "names": OUR_CLASS_NAMES,
        }, f, default_flow_style=False, sort_keys=False)
    print("Wrote data.yaml")
else:
    print("Dataset already loaded from zip; skipping filter step.")

### 1.3 — Merge the Kaggle engine-bay source

The Kaggle engine-bay dataset is already in YOLO format but uses its own class IDs over a
larger label set. We map only the categories that overlap our taxonomy onto our IDs
(`BATTERY->battery`, `ALTERNATOR->alternator`, `RADIATOR->radiator`, `OIL FILTER->oil_filter`)
and skip the rest. These are the cluttered, in-scene examples of the large components, so they
are added to the training split. Filenames are prefixed with `eb_` to avoid collisions.
Upload the Kaggle dataset to Drive and set `ENGINE_BAY_DIR` to its location.

In [ ]:
if not DATASET_READY:
    # Kaggle engine-bay class id -> our class id (None = skip, not one of our eight).
    ENGINE_BAY_TO_OURS = {
        1: 1,     # BATTERY     -> battery
        11: 0,    # ALTERNATOR  -> alternator
        13: 2,    # RADIATOR    -> radiator
        25: 6,    # OIL FILTER  -> oil_filter
        14: None, # AIR FILTER  -> skip (not in our eight)
    }
    # Upload the Kaggle engine-bay dataset to Drive and set this to its folder.
    ENGINE_BAY_DIR = "/content/drive/MyDrive/engine_bay_kaggle"

    def find_engine_bay_paths(root):
        img_dir = lbl_dir = None
        for r, _, files in os.walk(root):
            base = os.path.basename(r)
            if base == "images" and any(f.lower().endswith((".jpg", ".png")) for f in files):
                img_dir = r
            if base == "labels" and any(f.endswith(".txt") for f in files):
                lbl_dir = r
        return img_dir, lbl_dir

    def remap_engine_bay_label(label_path):
        out = []
        with open(label_path) as f:
            for line in f:
                parts = line.strip().split()
                if len(parts) < 5:
                    continue
                old = int(parts[0])
                if ENGINE_BAY_TO_OURS.get(old) is not None:
                    out.append(f"{ENGINE_BAY_TO_OURS[old]} {' '.join(parts[1:])}")
        return out

    img_dir, lbl_dir = find_engine_bay_paths(ENGINE_BAY_DIR)
    print(f"Engine-bay images: {img_dir}")
    print(f"Engine-bay labels: {lbl_dir}")

    dst_images = os.path.join(DATASET_DIR, "images", "train")
    dst_labels = os.path.join(DATASET_DIR, "labels", "train")
    os.makedirs(dst_images, exist_ok=True)
    os.makedirs(dst_labels, exist_ok=True)

    added = 0
    for label_file in os.listdir(lbl_dir):
        if not label_file.endswith(".txt"):
            continue
        remapped = remap_engine_bay_label(os.path.join(lbl_dir, label_file))
        if not remapped:
            continue
        stem = os.path.splitext(label_file)[0]
        for ext in [".jpg", ".jpeg", ".png", ".webp"]:
            img = os.path.join(img_dir, stem + ext)
            if os.path.exists(img):
                shutil.copy2(img, os.path.join(dst_images, f"eb_{stem}{ext}"))
                with open(os.path.join(dst_labels, f"eb_{label_file}"), "w") as f:
                    f.write("\n".join(remapped) + "\n")
                added += 1
                break
    print(f"Added {added} engine-bay images to the training split")
else:
    print("Dataset already loaded from zip; skipping Kaggle merge.")

### 1.4 — Repack the freshly built dataset into `dataset.zip`

If you built the dataset from scratch in 1.1–1.3, this cell zips the result and saves it to
Drive as `dataset.zip`, so future runs take the fast path in 1.0. Skip if the dataset was
loaded from a prebuilt zip.

The repository also includes a Selenium-based scraper (CarParts.com) and a cleaning script
used to gather additional images per class during development; that path is optional and is
documented in the repository rather than run here.

In [ ]:
if not DATASET_READY:
    archive = shutil.make_archive("/content/dataset", "zip", DATASET_DIR)
    shutil.copy(archive, ZIP_PATH)
    print(f"Built dataset zipped and saved to {ZIP_PATH}")
    for split in ["train", "val", "test"]:
        img_dir = os.path.join(DATASET_DIR, "images", split)
        if os.path.exists(img_dir):
            n = len([f for f in os.listdir(img_dir) if f.lower().endswith((".jpg", ".jpeg", ".png"))])
            print(f"  {split:6s}: {n} images")
else:
    print("Dataset was loaded from prebuilt zip; nothing to repack.")

## Part 2 — Leakage Check and Clean Split

Merging two public sources risks near-duplicate leakage: the same component photographed
under slightly different conditions, or the same image re-encoded, can end up on both sides
of the train/test boundary, making the reported test accuracy measure memorisation rather
than generalisation. We detect this with perceptual hashing, then re-split the data by
source identity so that no source appears in more than one partition.

### 2.1 — Detect near-duplicate leakage with perceptual hashing

Each image is reduced to a 64-bit perceptual hash (8x8 hash size). Two images are treated as
near-duplicates when their Hamming distance is at most five bits. We count how many such pairs
cross split boundaries and, more importantly, how many *sources* appear in more than one split.
A non-zero source leakage means the split must be rebuilt.

In [ ]:
from pathlib import Path
import imagehash
from PIL import Image
import itertools, re

ROOT = Path(DATASET_DIR)
SPLITS = {'train': 'images/train', 'val': 'images/val', 'test': 'images/test'}
THRESH = 5
EXTS = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}

data = {}
for split, rel in SPLITS.items():
    folder = ROOT / rel
    entries = []
    if folder.exists():
        for p in folder.rglob('*'):
            if p.suffix.lower() in EXTS:
                try:
                    with Image.open(p) as im:
                        entries.append((p.name, imagehash.phash(im)))
                except Exception as e:
                    print(f'  ! unreadable {p.name}: {e}')
    data[split] = entries
    print(f'{split:5s}: {len(entries)} images hashed')

print('\n--- cross-split near-duplicate pairs (pHash) ---')
hits = []
for a, b in itertools.combinations(data.keys(), 2):
    for na, ha in data[a]:
        for nb, hb in data[b]:
            if (ha - hb) <= THRESH:
                hits.append((a, na, b, nb))
print(f'  {len(hits)} cross-split pHash pairs')

def source_of(fname):
    m = re.split(r'_jpg\.rf\.', fname, maxsplit=1)
    return m[0] if len(m) == 2 else fname

src_split = {}
for split, entries in data.items():
    for name, _ in entries:
        src_split.setdefault(source_of(name), set()).add(split)

leaked = {s: sp for s, sp in src_split.items() if len(sp) > 1}
total = len(src_split)
print(f'\n--- per-source check ---')
print(f'  distinct sources: {total}')
print(f'  sources in >1 split (LEAKED): {len(leaked)} ({100*len(leaked)/max(total,1):.1f}%)')
if leaked:
    print('  => split must be rebuilt by source (see next cell)')
else:
    print('  => clean split, you can proceed to training')

### 2.2 — Re-split by source identity

We group every image by its source and assign whole groups to train/validation/test in a
70/15/15 ratio with a fixed seed (42). Because all augmented variants of a photograph share a
source, they stay together in one split. The rebuilt dataset is written to a new folder with
an updated `data.yaml`.

In [ ]:
import random
from collections import defaultdict

random.seed(42)
SRC = DATASET_DIR
DST = '/content/data/dataset_clean'
RATIOS = (0.70, 0.15, 0.15)

def source_of(fname):
    m = re.split(r'_jpg\.rf\.', fname, maxsplit=1)
    return m[0] if len(m) == 2 else os.path.splitext(fname)[0]

pool = defaultdict(list)
for split in ['train', 'val', 'test']:
    img_dir = os.path.join(SRC, 'images', split)
    lbl_dir = os.path.join(SRC, 'labels', split)
    if not os.path.isdir(img_dir):
        continue
    for f in os.listdir(img_dir):
        if not f.lower().endswith(('.jpg', '.jpeg', '.png')):
            continue
        stem = os.path.splitext(f)[0]
        lbl = os.path.join(lbl_dir, stem + '.txt')
        pool[source_of(f)].append((os.path.join(img_dir, f), lbl))

sources = list(pool.keys())
random.shuffle(sources)
n = len(sources)
n_tr = int(n * RATIOS[0])
n_va = int(n * RATIOS[1])
assign = {}
for i, s in enumerate(sources):
    assign[s] = 'train' if i < n_tr else ('val' if i < n_tr + n_va else 'test')

for split in ['train', 'val', 'test']:
    os.makedirs(os.path.join(DST, 'images', split), exist_ok=True)
    os.makedirs(os.path.join(DST, 'labels', split), exist_ok=True)

counts = defaultdict(int)
for s, items in pool.items():
    split = assign[s]
    for img_path, lbl_path in items:
        shutil.copy(img_path, os.path.join(DST, 'images', split, os.path.basename(img_path)))
        if os.path.exists(lbl_path):
            shutil.copy(lbl_path, os.path.join(DST, 'labels', split, os.path.basename(lbl_path)))
        counts[split] += 1

print(f"Re-split over {n} distinct sources:")
for split in ['train', 'val', 'test']:
    print(f"  {split:6s}: {counts[split]} images")

import yaml
with open(os.path.join(SRC, 'data.yaml')) as f:
    y = yaml.safe_load(f)
y['path'] = DST
y['train'] = 'images/train'
y['val'] = 'images/val'
y['test'] = 'images/test'
with open(os.path.join(DST, 'data.yaml'), 'w') as f:
    yaml.dump(y, f, default_flow_style=False, sort_keys=False)
print("\nWrote data.yaml in dataset_clean/")

### 2.3 — Select the dataset and report final counts

From here on we use the clean, re-split dataset if it exists. The final per-split counts
should match the paper: 2013 / 440 / 445 for train / val / test.

In [ ]:
DATASET = '/content/data/dataset_clean' if os.path.exists('/content/data/dataset_clean/data.yaml') else DATASET_DIR
DATA_YAML = os.path.join(DATASET, 'data.yaml')
print(f"Using dataset: {DATASET}")
for split in ['train', 'val', 'test']:
    img_dir = os.path.join(DATASET, 'images', split)
    if os.path.exists(img_dir):
        n = len([f for f in os.listdir(img_dir) if f.lower().endswith(('.jpg', '.jpeg', '.png'))])
        print(f"  {split:6s}: {n} images")

## Part 3 — Training the Detector

The primary detector is YOLOv11m, fine-tuned from COCO-pretrained weights rather than trained
from scratch, since the dataset is too small to support training from random initialisation.
Training runs for up to 100 epochs with early stopping (patience 15), batch size 16 and input
resolution 640, with a fixed seed (42) so the later variant comparison and ablation differ
only in the factor under study. Augmentation combines mosaic (closed in the final epochs),
HSV jitter and geometric transforms.

In [ ]:
from ultralytics import YOLO

# Fine-tune YOLOv11m from COCO-pretrained weights.
model = YOLO("yolo11m.pt")
results = model.train(
    data=DATA_YAML,
    epochs=100,
    patience=15,
    batch=16,
    imgsz=640,
    seed=42,
    optimizer="AdamW",
    project="/content/drive/MyDrive/yolo_training",
    name="engine_components",
    exist_ok=True,
)

## Part 4 — Evaluation

With the model trained, we evaluate it on the held-out test set. We report the overall
metrics (mAP@0.5, mAP@0.5:0.95, precision, recall), then break performance down per class to
see where the detector is strong and where it struggles, and finally inspect the confusion
matrix to understand the structure of the errors.

### 4.1 — Overall test metrics

We run validation on the `test` split. The summary metrics are the standard object-detection
measures: mean average precision at an IoU threshold of 0.5 (mAP@0.5), averaged over
thresholds from 0.5 to 0.95 (mAP@0.5:0.95), plus precision and recall.

In [ ]:
# Evaluate the trained model on the held-out test split.
metrics = model.val(data=DATA_YAML, split='test')
print(f"mAP@0.5      : {metrics.box.map50:.3f}")
print(f"mAP@0.5:0.95 : {metrics.box.map:.3f}")
print(f"Precision    : {metrics.box.mp:.3f}")
print(f"Recall       : {metrics.box.mr:.3f}")

### 4.2 — Locate the run directory

Ultralytics writes all evaluation artefacts (plots, confusion matrix, curves) to a run
directory. We capture its path so the following cells can display the generated figures.

In [ ]:
# The validation results directory holds the generated plots and curves.
import os
RUN_DIR = metrics.save_dir
print("Run directory:", RUN_DIR)
print("Files:")
for f in sorted(os.listdir(RUN_DIR)):
    print("  ", f)

### 4.3 — Per-class performance

The overall figure hides a wide spread across classes. This table lists precision, recall and
mAP for each of the eight components, ordered from best to worst, so we can see which parts the
detector handles reliably and which it does not.

In [ ]:
# Per-class metrics, sorted from best to worst by mAP@0.5.
import pandas as pd

names = model.names  # id -> class name
rows = []
for i, c in enumerate(metrics.box.ap_class_index):
    rows.append({
        "class": names[c],
        "precision": round(metrics.box.p[i], 3),
        "recall": round(metrics.box.r[i], 3),
        "mAP@0.5": round(metrics.box.ap50[i], 3),
        "mAP@0.5:0.95": round(metrics.box.ap[i], 3),
    })

df = pd.DataFrame(rows).sort_values("mAP@0.5", ascending=False).reset_index(drop=True)
print(df.to_string(index=False))

### 4.4 — Confusion matrix

The normalised confusion matrix shows where misclassifications go. The key question is whether
the weak classes are confused with *other components* (an inter-class problem) or lost to the
*background* (a missed-detection problem). For this detector, the large in-scene components
(radiator, battery) lose mass to the background row, which means the dominant error is missed
detection in cluttered scenes, not class confusion.

In [ ]:
# Display the normalised confusion matrix produced during validation.
from IPython.display import Image, display

def show(fname):
    path = os.path.join(RUN_DIR, fname)
    if os.path.exists(path):
        display(Image(filename=path))
    else:
        print("Not found:", path)

show('confusion_matrix_normalized.png')

### 4.5 — Training curves and PR curve

Finally we display the training curves and the precision-recall curve. Healthy training shows
the losses falling and then flattening, with validation metrics plateauing rather than
diverging, which indicates the model has learned general features rather than overfitting the
small training set.

In [ ]:
# Display the training results curves and the F1 / PR curves.
show('results.png')
show('F1_curve.png')
show('PR_curve.png')

## Part 5 — In-Domain Variant Comparison

Cross-version YOLO benchmarks are usually reported on generic datasets such as MS COCO, which
need not predict the ranking on a narrow target. To find out which variant is actually best
for this problem, we train the nano, small and medium variants under identical conditions and
compare them on the same test set. Holding everything but model capacity fixed isolates the
effect of size.

### 5.1 — Setup

We define the three variants to compare and a separate output directory for the comparison
runs, keeping them apart from the main training run above.

In [ ]:
# Variant comparison setup.
import os, time, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from ultralytics import YOLO

COMPARE_DIR = '/content/drive/MyDrive/yolo_training/comparison'
os.makedirs(COMPARE_DIR, exist_ok=True)

DATA_YAML = os.path.join(DATASET, 'data.yaml')
class_names = ['alternator', 'battery', 'radiator', 'ignition_coil',
               'starter', 'fuel_injector', 'oil_filter', 'spark_plug']

VARIANTS = ['yolo11n', 'yolo11s', 'yolo11m']
print(f"Dataset: {DATASET}")
print(f"Variants to train: {VARIANTS}")

### 5.2 — Train the three variants

Each variant is trained from its COCO-pretrained weights under identical settings (100 epochs,
batch 16, image size 640, patience 15, seed 42), so the only thing that varies between runs is
model capacity. Already-trained variants are skipped, so the cell can be re-run safely.

In [ ]:
# Train n/s/m under identical conditions. Skip any variant already trained.
for variant in VARIANTS:
    out_name = f'cmp_{variant}'
    weights_path = f'/content/drive/MyDrive/yolo_training/{out_name}/weights/best.pt'
    if os.path.exists(weights_path):
        print(f"[skip] {variant} already trained: {weights_path}")
        continue
    print(f"\n{'='*50}\nTraining {variant}\n{'='*50}")
    m = YOLO(f'{variant}.pt')
    m.train(
        data=DATA_YAML,
        epochs=100,
        imgsz=640,
        batch=16,
        name=out_name,
        patience=15,
        seed=42,                   # same seed across variants for a fair comparison
        project='/content/drive/MyDrive/yolo_training',
        plots=True,
    )
    print(f"[done] {variant}")

### 5.3 — Evaluate each variant

For every trained variant we measure test-set accuracy (mAP, precision, recall), inference
speed (averaged over 100 test images), model size and parameter count, and the per-class
mAP@0.5. Results are cached to JSON so the comparison can be rebuilt without re-evaluating.

In [ ]:
# Evaluate every variant: accuracy, speed, size, per-class mAP.
results_all = {}
for variant in VARIANTS:
    weights = f'/content/drive/MyDrive/yolo_training/cmp_{variant}/weights/best.pt'
    if not os.path.exists(weights):
        print(f"[missing] {variant} - train it in the previous cell")
        continue
    m = YOLO(weights)
    metr = m.val(data=DATA_YAML, split='test', verbose=False)

    # Inference speed: mean over up to 100 test images.
    test_dir = os.path.join(DATASET, 'images', 'test')
    imgs = [os.path.join(test_dir, f) for f in os.listdir(test_dir)
            if f.lower().endswith(('.jpg', '.png'))][:100]
    t = []
    for p in imgs:
        s = time.perf_counter()
        m.predict(p, conf=0.25, verbose=False)
        t.append((time.perf_counter() - s) * 1000)

    file_mb = os.path.getsize(weights) / 1e6
    results_all[variant] = {
        'mAP50': metr.box.map50,
        'mAP50-95': metr.box.map,
        'precision': metr.box.mp,
        'recall': metr.box.mr,
        'ms_per_img': float(np.mean(t)),
        'fps': 1000 / float(np.mean(t)),
        'params_M': sum(p.numel() for p in m.model.parameters()) / 1e6,
        'file_MB': file_mb,
        'per_class_ap50': metr.box.ap50.tolist(),
    }
    print(f"{variant}: mAP50={metr.box.map50:.3f} mAP50-95={metr.box.map:.3f} "
          f"FPS={results_all[variant]['fps']:.1f}")

with open(f'{COMPARE_DIR}/results_all.json', 'w') as f:
    json.dump(results_all, f, indent=2)
print(f"\nSaved: {COMPARE_DIR}/results_all.json")

### 5.4 — Comparison table

This is the headline comparison: parameters, file size, accuracy and speed side by side. The
expected finding is that the small variant matches the medium one while being far smaller, so
extra capacity brings no benefit on this dataset.

In [ ]:
# Build the comparison table.
rows = []
for v in VARIANTS:
    if v not in results_all:
        continue
    r = results_all[v]
    rows.append({
        'Model': v.replace('yolo11', 'YOLOv11'),
        'Params (M)': round(r['params_M'], 1),
        'File (MB)': round(r['file_MB'], 1),
        'mAP50': round(r['mAP50'], 3),
        'mAP50-95': round(r['mAP50-95'], 3),
        'Precision': round(r['precision'], 3),
        'Recall': round(r['recall'], 3),
        'ms/img': round(r['ms_per_img'], 1),
        'FPS': round(r['fps'], 1),
    })
cmp_df = pd.DataFrame(rows)
print(cmp_df.to_string(index=False))
cmp_df.to_csv(f'{COMPARE_DIR}/model_comparison.csv', index=False)

### 5.5 — Comparison charts

Two views of the same result: accuracy per variant on the left, and the accuracy-versus-speed
trade-off on the right. When the small and medium variants sit at the same accuracy, the
smaller and faster one is preferable.

In [ ]:
# Accuracy bars and accuracy-vs-speed trade-off.
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

names = [r['Model'] for r in rows]
map50 = [r['mAP50'] for r in rows]
map5095 = [r['mAP50-95'] for r in rows]
fps = [r['FPS'] for r in rows]

x = np.arange(len(names)); w = 0.35
ax1.bar(x - w/2, map50, w, label='mAP50', color='#FF9800')
ax1.bar(x + w/2, map5095, w, label='mAP50-95', color='#4CAF50')
ax1.set_xticks(x); ax1.set_xticklabels(names)
ax1.set_ylabel('mAP'); ax1.set_title('Accuracy per model', fontweight='bold')
ax1.legend(); ax1.grid(axis='y', alpha=0.3); ax1.set_ylim(0, 1)
for i, (a, b) in enumerate(zip(map50, map5095)):
    ax1.text(i - w/2, a + 0.01, f'{a:.2f}', ha='center', fontsize=9)
    ax1.text(i + w/2, b + 0.01, f'{b:.2f}', ha='center', fontsize=9)

ax2.scatter(fps, map50, s=200, c=['#3498db', '#2ecc71', '#e74c3c'][:len(names)], zorder=3)
for n, f, a in zip(names, fps, map50):
    ax2.annotate(n, (f, a), textcoords="offset points", xytext=(0, 12), ha='center', fontsize=10)
ax2.set_xlabel('FPS'); ax2.set_ylabel('mAP50')
ax2.set_title('Accuracy vs speed trade-off', fontweight='bold')
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.savefig(f'{COMPARE_DIR}/model_comparison_charts.png', dpi=150)
plt.show()

### 5.6 — Per-class mAP across variants

Finally, a per-class breakdown across the three variants, as a table and a heatmap. The easy
classes saturate near the top for all three variants, so any capacity difference can only show
up on the hard in-scene classes, where the gains are within run-to-run variance.

In [ ]:
# Per-class mAP@0.5 for each variant, as a table and a heatmap.
per_class_rows = []
for v in VARIANTS:
    if v not in results_all:
        continue
    ap = results_all[v]['per_class_ap50']
    row = {'Model': v.replace('yolo11', 'YOLOv11')}
    for name, val in zip(class_names, ap):
        row[name] = round(val, 3)
    per_class_rows.append(row)
pc_df = pd.DataFrame(per_class_rows)
print(pc_df.to_string(index=False))
pc_df.to_csv(f'{COMPARE_DIR}/per_class_per_model.csv', index=False)

fig, ax = plt.subplots(figsize=(14, 4))
mat = np.array([[results_all[v]['per_class_ap50'][i] for i in range(8)]
                for v in VARIANTS if v in results_all])
im = ax.imshow(mat, cmap='RdYlGn', vmin=0.5, vmax=1.0, aspect='auto')
ax.set_xticks(range(8)); ax.set_xticklabels(class_names, rotation=45, ha='right')
ax.set_yticks(range(len(mat)))
ax.set_yticklabels([v.replace('yolo11', 'YOLOv11') for v in VARIANTS if v in results_all])
for i in range(mat.shape[0]):
    for j in range(mat.shape[1]):
        ax.text(j, i, f'{mat[i, j]:.2f}', ha='center', va='center', fontsize=9)
plt.colorbar(im, label='mAP50')
ax.set_title('Per-class mAP50 per model', fontweight='bold')
plt.tight_layout()
plt.savefig(f'{COMPARE_DIR}/per_class_heatmap.png', dpi=150)
plt.show()

### 5.7 — Qualitative side-by-side

To see the variants in action, we run all three on the same few test images and display their
detections side by side. On isolated parts the variants agree; on cluttered engine-bay scenes
the differences appear, which is consistent with the quantitative finding that difficulty lies
in the in-scene components.

In [ ]:
# Run all variants on the same few test images and show detections side by side.
import random
test_dir = os.path.join(DATASET, 'images', 'test')
sample_imgs = random.sample([f for f in os.listdir(test_dir)
                             if f.lower().endswith(('.jpg', '.png'))], 3)

models_loaded = {v: YOLO(f'/content/drive/MyDrive/yolo_training/cmp_{v}/weights/best.pt')
                 for v in VARIANTS
                 if os.path.exists(f'/content/drive/MyDrive/yolo_training/cmp_{v}/weights/best.pt')}

fig, axes = plt.subplots(len(sample_imgs), len(models_loaded),
                         figsize=(6 * len(models_loaded), 5 * len(sample_imgs)))
if len(sample_imgs) == 1:
    axes = axes[None, :]

for r, img_name in enumerate(sample_imgs):
    img_path = os.path.join(test_dir, img_name)
    for c, (v, m) in enumerate(models_loaded.items()):
        res = m.predict(img_path, conf=0.25, verbose=False)
        axes[r, c].imshow(res[0].plot()[:, :, ::-1])
        axes[r, c].axis('off')
        if r == 0:
            axes[r, c].set_title(v.replace('yolo11', 'YOLOv11'), fontsize=12, fontweight='bold')
        axes[r, c].text(0.02, 0.98, f'{len(res[0].boxes)} det',
                        transform=axes[r, c].transAxes, fontsize=9,
                        va='top', bbox=dict(boxstyle='round', fc='white', alpha=0.7))

plt.suptitle('Same images, three models', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{COMPARE_DIR}/side_by_side.png', dpi=150)
plt.show()

## Part 6 — Augmentation Ablation

Knowing which augmentations actually help is as useful as knowing which model to use. We test
this directly: starting from the full augmentation set, we remove one family at a time
(mosaic, photometric HSV jitter, geometric transforms), retrain YOLOv11m under otherwise
identical conditions, and measure the drop. A negative change means that augmentation was
contributing; a small positive change is within run-to-run noise and is treated as
inconclusive.

### 6.1 — Define the ablation configurations

Each configuration changes exactly one augmentation family relative to the full baseline, so
the effect of that family can be isolated. All runs use the same model (m), the same seed and
the same number of epochs.

In [ ]:
# Augmentation ablation on YOLOv11m: remove one family at a time.
import os, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from ultralytics import YOLO

ABLATION_DIR = '/content/drive/MyDrive/yolo_training/ablation'
os.makedirs(ABLATION_DIR, exist_ok=True)
DATA_YAML = os.path.join(DATASET, 'data.yaml')

# Each config changes ONE augmentation family from the baseline.
# All use the same model (m), same seed, same number of epochs.
ABLATION_CONFIGS = {
    'full':         {},                                                       # all augmentations on (baseline)
    'no_mosaic':    {'mosaic': 0.0},
    'no_hsv':       {'hsv_h': 0.0, 'hsv_s': 0.0, 'hsv_v': 0.0},
    'no_geometric': {'degrees': 0.0, 'translate': 0.0, 'scale': 0.0, 'fliplr': 0.0},
}
ABLATION_EPOCHS = 100
print(f"Ablation configs: {list(ABLATION_CONFIGS.keys())}")
print(f"Epochs per config: {ABLATION_EPOCHS}")

### 6.2 — Train each ablation configuration

Each configuration is trained from scratch (from COCO weights) with the one augmentation family
disabled. The seed and epoch count are identical across all configurations, which is essential
for the comparison to be fair. Already-trained configs are skipped.

In [ ]:
# Train each ablation config. Same seed and epochs across all of them.
for cfg_name, overrides in ABLATION_CONFIGS.items():
    out_name = f'abl_{cfg_name}'
    weights = f'/content/drive/MyDrive/yolo_training/{out_name}/weights/best.pt'
    if os.path.exists(weights):
        print(f"[skip] {cfg_name} already trained")
        continue
    print(f"\n{'='*50}\nAblation: {cfg_name}  overrides={overrides}\n{'='*50}")
    m = YOLO('yolo11m.pt')
    m.train(
        data=DATA_YAML,
        epochs=ABLATION_EPOCHS,
        imgsz=640,
        batch=16,
        name=out_name,
        patience=15,
        seed=42,                 # identical across all configs - essential
        project='/content/drive/MyDrive/yolo_training',
        plots=True,
        **overrides,
    )
    print(f"[done] {cfg_name}")

### 6.3 — Evaluate each configuration

Every ablation model is evaluated on the same test set, recording the summary metrics. Results
are cached to JSON.

In [ ]:
# Evaluate every ablation config on the test set.
ablation_results = {}
test_dir = os.path.join(DATASET, 'images', 'test')

for cfg_name in ABLATION_CONFIGS:
    weights = f'/content/drive/MyDrive/yolo_training/abl_{cfg_name}/weights/best.pt'
    if not os.path.exists(weights):
        print(f"[missing] {cfg_name}")
        continue
    m = YOLO(weights)
    metr = m.val(data=DATA_YAML, split='test', verbose=False)
    ablation_results[cfg_name] = {
        'mAP50': metr.box.map50,
        'mAP50-95': metr.box.map,
        'precision': metr.box.mp,
        'recall': metr.box.mr,
    }
    print(f"{cfg_name:14s}: mAP50={metr.box.map50:.4f}  mAP50-95={metr.box.map:.4f}")

with open(f'{ABLATION_DIR}/ablation_results.json', 'w') as f:
    json.dump(ablation_results, f, indent=2)

### 6.4 — Ablation table

For each configuration we report the absolute mAP and the change relative to the full baseline.
A negative delta means removing that augmentation hurt performance, so the augmentation helps.
A small positive delta (within roughly 1-2 points) is within run-to-run variance and is
inconclusive rather than an improvement.

In [ ]:
# Ablation table: absolute metrics and delta vs the full baseline.
if 'full' in ablation_results:
    base = ablation_results['full']
    rows = []
    for cfg, r in ablation_results.items():
        rows.append({
            'Config': cfg,
            'mAP50': round(r['mAP50'], 4),
            'd_mAP50': round(r['mAP50'] - base['mAP50'], 4) if cfg != 'full' else 0.0,
            'mAP50-95': round(r['mAP50-95'], 4),
            'd_mAP50-95': round(r['mAP50-95'] - base['mAP50-95'], 4) if cfg != 'full' else 0.0,
        })
    abl_df = pd.DataFrame(rows)
    print(abl_df.to_string(index=False))
    abl_df.to_csv(f'{ABLATION_DIR}/ablation_table.csv', index=False)
    print("\nReading: a negative delta means removing that augmentation HURTS performance,")
    print("so the augmentation helps. A small positive delta (<1-2%) is within variance.")
else:
    print("Missing 'full' baseline - run the training cell first.")

### 6.5 — Ablation chart

The same deltas as a bar chart: red bars (negative) mark augmentations that help, green bars
(positive) mark ones whose removal did not hurt. The largest negative bar identifies the single
most valuable augmentation family.

In [ ]:
# Bar chart of the delta from removing each augmentation family.
if 'full' in ablation_results:
    cfgs = [c for c in ABLATION_CONFIGS if c in ablation_results]
    deltas = [ablation_results[c]['mAP50'] - ablation_results['full']['mAP50'] for c in cfgs]
    colors = ['#888' if c == 'full' else ('#e74c3c' if d < 0 else '#2ecc71')
              for c, d in zip(cfgs, deltas)]
    fig, ax = plt.subplots(figsize=(10, 5))
    ax.bar(cfgs, deltas, color=colors)
    ax.axhline(0, color='black', linewidth=0.8)
    ax.set_ylabel('d mAP50 vs baseline (full)')
    ax.set_title('Impact of removing each augmentation', fontweight='bold')
    ax.grid(axis='y', alpha=0.3)
    for i, d in enumerate(deltas):
        ax.text(i, d + (0.002 if d >= 0 else -0.004), f'{d:+.3f}', ha='center', fontsize=9)
    plt.tight_layout()
    plt.savefig(f'{ABLATION_DIR}/ablation_chart.png', dpi=150)
    plt.show()

### 6.6 — Summary: model and augmentation choices

This final cell pulls together the two studies: which variant is best from the comparison, and
which augmentations matter from the ablation. It states the conclusions honestly, noting that
differences within run-to-run variance should not be presented as strong evidence.

In [ ]:
# Pull together the conclusions from the comparison and the ablation.
print("=" * 60)
print("CONCLUSION: MODEL AND AUGMENTATION CHOICES")
print("=" * 60)

# 1. Best model from the comparison.
if 'results_all' in dir() and results_all:
    print("\n--- Variant comparison ---")
    best_acc = max(results_all, key=lambda v: results_all[v]['mAP50'])
    fastest = max(results_all, key=lambda v: results_all[v]['fps'])
    for v in VARIANTS:
        if v in results_all:
            r = results_all[v]
            print(f"  {v.replace('yolo11', 'YOLOv11'):12s}: "
                  f"mAP50={r['mAP50']:.3f}, FPS={r['fps']:.0f}, {r['params_M']:.1f}M params")
    print(f"\n  Most accurate: {best_acc.replace('yolo11', 'YOLOv11')}")
    print(f"  Fastest:       {fastest.replace('yolo11', 'YOLOv11')}")
    print("\n  On this dataset the small variant matches the medium one, so the")
    print("  smaller, faster model is preferable: a useful negative result.")

# 2. Which augmentations matter.
if 'ablation_results' in dir() and 'full' in ablation_results:
    print("\n--- Augmentation ablation ---")
    base = ablation_results['full']['mAP50']
    for cfg, r in ablation_results.items():
        if cfg == 'full':
            continue
        d = r['mAP50'] - base
        verdict = "helps" if d < -0.01 else ("negligible" if abs(d) <= 0.01 else "possible noise")
        print(f"  removing '{cfg.replace('no_', '')}': d={d:+.3f} mAP50 -> {verdict}")
    print("\n  Honest note: differences under ~1-2% are within run-to-run variance")
    print("  and should not be presented as strong evidence.")

## Part 7 — Quantifying the Effect of Leakage

The whole dataset pipeline rests on the per-source de-duplication that keeps near-duplicates
out of the test set. This final experiment measures how much that step is worth. We
deliberately contaminate a copy of the clean dataset by copying 40% of the test images back
into training, train the same model on both the clean and the contaminated data, and evaluate
both on the *same clean test set*. Any difference is purely the inflation caused by leakage.

### 7.1 — Setup

We point the experiment at the clean dataset built in Part 2 and define where the contaminated
copy will be created. The pHash threshold matches the one used for de-duplication, so the
leakage count is measured on the same basis.

In [ ]:
# Leakage on/off experiment: setup.
import os, shutil, random, itertools
from pathlib import Path
from PIL import Image
import imagehash
from ultralytics import YOLO

CLEAN_ROOT = Path('/content/data/dataset_clean')   # the clean (de-duplicated) dataset
LEAK_ROOT  = Path('/content/data/dataset_leaky')   # created automatically below
EPOCHS     = 100        # lower this for a faster run
SEED       = 42
THRESH     = 5          # same pHash threshold as the de-duplication step
PROJECT    = '/content/drive/MyDrive/yolo_training'
SPLITS     = {'train': 'images/train', 'val': 'images/val', 'test': 'images/test'}
EXTS       = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}
random.seed(42)

assert (CLEAN_ROOT / 'data.yaml').exists(), \
    f"Cannot find {CLEAN_ROOT}/data.yaml - run the cells that build dataset_clean first"
print("Config OK. CLEAN_ROOT =", CLEAN_ROOT)

### 7.2 — Inject artificial leakage

We copy the clean dataset and then deliberately leak it: 40% of the test images (with their
labels) are copied into the training set, prefixed with `leak_` so they are easy to identify.
This simulates the contamination that an uncorrected merge would have produced.

In [ ]:
# Build the contaminated variant (leakage ON) by copying 40% of test into train.
print(">>> Building the contaminated variant (leakage ON)...")
if LEAK_ROOT.exists():
    shutil.rmtree(LEAK_ROOT)
shutil.copytree(CLEAN_ROOT, LEAK_ROOT)

test_img  = LEAK_ROOT / 'images/test'
test_lbl  = LEAK_ROOT / 'labels/test'
train_img = LEAK_ROOT / 'images/train'
train_lbl = LEAK_ROOT / 'labels/train'

test_files = [p for p in test_img.rglob('*') if p.suffix.lower() in EXTS]
n_leak = int(len(test_files) * 0.40)              # 40% of test -> train
leaked = random.sample(test_files, n_leak)
count = 0
for p in leaked:
    shutil.copy(p, train_img / f'leak_{p.name}')
    lbl = test_lbl / (p.stem + '.txt')
    if lbl.exists():
        shutil.copy(lbl, train_lbl / f'leak_{p.stem}.txt')
    count += 1
print(f"    Injected {count} images from test into train.")

clean_yaml = (CLEAN_ROOT / 'data.yaml').read_text()
(LEAK_ROOT / 'data.yaml').write_text(clean_yaml.replace(str(CLEAN_ROOT), str(LEAK_ROOT)))
print("    Wrote data.yaml for the leaky variant.")

### 7.3 — Confirm the contamination with pHash

Before training, we verify that the contamination actually increased cross-split near-duplicate
overlap. The clean dataset should show essentially no cross-split duplicates, while the leaky
one should show many, confirming that the injection worked.

In [ ]:
# Confirm how many cross-split near-duplicates each variant has.
def count_cross_split_dupes(root):
    data = {}
    for split, rel in SPLITS.items():
        folder = root / rel
        entries = []
        if folder.exists():
            for p in folder.rglob('*'):
                if p.suffix.lower() in EXTS:
                    try:
                        with Image.open(p) as im:
                            entries.append((p.name, imagehash.phash(im)))
                    except Exception:
                        pass
        data[split] = entries
    hits = 0
    for a, b in itertools.combinations(data.keys(), 2):
        for _, ha in data[a]:
            for _, hb in data[b]:
                if (ha - hb) <= THRESH:
                    hits += 1
    return hits

print("Cross-split near-duplicates CLEAN:", count_cross_split_dupes(CLEAN_ROOT))
print("Cross-split near-duplicates LEAKY:", count_cross_split_dupes(LEAK_ROOT))

### 7.4 — Train clean and leaky, evaluate on the same clean test set

The key to a fair comparison: both models are trained under identical settings, but *both are
evaluated on the same clean test set*. The leaky model has seen some of those test images during
training, so any gain it shows is inflation, not genuine improvement.

In [ ]:
# Train both variants and evaluate both on the SAME clean test set.
def train_and_eval(data_yaml, name):
    print(f"\n>>> Training '{name}' ...")
    model = YOLO('yolo11m.pt')
    model.train(
        data=data_yaml, epochs=EPOCHS, imgsz=640, batch=16,
        name=name, patience=15, seed=SEED, save=True,
        project=PROJECT, plots=True, verbose=False,
    )
    # Both evaluated on the SAME clean test set.
    m = model.val(data=str(CLEAN_ROOT / 'data.yaml'), split='test')
    return {'mAP50': float(m.box.map50), 'mAP50-95': float(m.box.map),
            'P': float(m.box.mp), 'R': float(m.box.mr)}

clean_res = train_and_eval(str(CLEAN_ROOT / 'data.yaml'), 'leak_off_clean')
leak_res  = train_and_eval(str(LEAK_ROOT  / 'data.yaml'), 'leak_on_contaminated')

### 7.5 — Report the inflation

Finally we tabulate the difference. The inflation column is the gap between the leaky and clean
models on every metric. A clear positive inflation confirms that uncorrected near-duplicate
leakage would have made the detector look substantially better than it is, which is the
empirical justification for the de-duplication behind the dataset.

In [ ]:
# Report leak ON vs leak OFF on the same clean test set.
print("=" * 55)
print("  LEAKAGE ON/OFF (evaluated on the same clean test set)")
print("=" * 55)
print(f"{'metric':12} {'leak OFF':>10} {'leak ON':>10} {'inflation':>10}")
for k in ['mAP50', 'mAP50-95', 'P', 'R']:
    off, on = clean_res[k], leak_res[k]
    print(f"{k:12} {off:10.4f} {on:10.4f} {on-off:+10.4f}")
print("=" * 55)